# 14 — Sentimento da transcrição

O indicador resume o tom como `positivo`, `neutro`, `negativo` ou `misto`. Pysentimiento é preferido quando disponível; o fallback lexical mantém o notebook executável e declara que o score é heurístico.

## Modelo opcional

O analisador é carregado apenas na primeira utilização. Assim, abrir os notebooks não dispara download nem ocupa memória sem necessidade.

In [ ]:
SENTIMENT_MODEL_NAME = "pysentimiento/bertweet-pt-sentiment"

_SENTIMENT_ANALYZER = None

_SENTIMENT_LOAD_ERROR = None

POSITIVE_SENTIMENT = {
    "gostei": 2.0, "otimo": 2.0, "excelente": 2.0,
    "satisfeito": 2.0, "satisfeita": 2.0, "satisfeitos": 2.0,
    "funciona bem": 2.0, "aprovado": 1.5, "bom": 1.0,
    "melhorou": 1.0, "interesse": 1.0,
}

NEGATIVE_SENTIMENT = {
    "insatisfeito": 2.0, "insatisfeita": 2.0, "insatisfeitos": 2.0,
    "problema": 1.0, "ruim": 2.0, "pessimo": 2.0,
    "cancelar": 2.0, "reclamacao": 1.5, "frustrado": 2.0,
    "nao funciona": 2.0, "falha": 1.0, "dificuldade": 1.0,
}


In [ ]:
def _load_sentiment_analyzer():
    global _SENTIMENT_ANALYZER, _SENTIMENT_LOAD_ERROR
    if _SENTIMENT_ANALYZER is not None:
        return _SENTIMENT_ANALYZER
    if _SENTIMENT_LOAD_ERROR is not None:
        raise RuntimeError("Pysentimiento indisponível.") from _SENTIMENT_LOAD_ERROR
    try:
        from pysentimiento import create_analyzer

        _SENTIMENT_ANALYZER = create_analyzer(task="sentiment", lang="pt")
        return _SENTIMENT_ANALYZER
    except Exception as error:
        _SENTIMENT_LOAD_ERROR = error
        raise RuntimeError("Pysentimiento indisponível.") from error

def _sentiment_probabilities(output: Any) -> dict[str, float]:
    probabilities = getattr(output, "probas", None)
    if not isinstance(probabilities, dict):
        raise ValueError("Saída inválida do modelo de sentimento.")
    normalized = {str(label).split(".")[-1].upper(): float(score) for label, score in probabilities.items()}
    if not {"POS", "NEG", "NEU"}.issubset(normalized):
        raise ValueError("O modelo de sentimento não devolveu POS, NEG e NEU.")
    return normalized

def _model_sentiment(transcription: str) -> dict[str, Any]:
    analyzer = _load_sentiment_analyzer()
    chunk_probabilities = [
        _sentiment_probabilities(analyzer.predict(chunk))
        for chunk in _word_chunks(transcription)
    ]
    averages = {
        label: sum(item[label] for item in chunk_probabilities) / len(chunk_probabilities)
        for label in ("POS", "NEG", "NEU")
    }
    has_positive = any(item["POS"] >= 0.65 for item in chunk_probabilities)
    has_negative = any(item["NEG"] >= 0.65 for item in chunk_probabilities)
    if has_positive and has_negative:
        label = "misto"
        score = min(averages["POS"] + averages["NEG"], 1.0)
    else:
        winner = max(averages, key=averages.get)
        label = {"POS": "positivo", "NEG": "negativo", "NEU": "neutro"}[winner]
        score = averages[winner]
    return {
        "label": label,
        "score": round(score, 6),
        "score_type": "model_probability",
        "engine": "pysentimiento",
        "model": SENTIMENT_MODEL_NAME,
    }


## Fallback e negação

O fallback soma sinais positivos e negativos e inverte ocorrências negadas. Evidência relevante nos dois sentidos produz `misto`, evitando esconder divergências entre trechos.

In [ ]:
def _lexical_sentiment(transcription: str) -> dict[str, Any]:
    normalized = f" {_normalize(transcription)} "
    positive_score = 0.0
    negative_score = 0.0
    for phrase, weight in POSITIVE_SENTIMENT.items():
        if f" nao {phrase} " in normalized:
            negative_score += weight
        elif f" {phrase} " in normalized:
            positive_score += weight
    for phrase, weight in NEGATIVE_SENTIMENT.items():
        if f" nao {phrase} " in normalized:
            positive_score += weight
        elif f" {phrase} " in normalized:
            negative_score += weight

    if positive_score and negative_score:
        label = "misto"
    elif positive_score:
        label = "positivo"
    elif negative_score:
        label = "negativo"
    else:
        label = "neutro"
    evidence = positive_score + negative_score
    return {
        "label": label,
        "score": round(min(evidence / 4.0, 1.0), 6),
        "score_type": "heuristic",
        "engine": "lexical_sentiment",
        "model": None,
    }


## Escolha transparente do mecanismo

`fallback` força regras locais; `full` exige o modelo; `auto` tenta o modelo e registra o tipo da falha antes de continuar com as regras.

In [ ]:
def _analyze_sentiment(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_sentiment(transcription), "fallback", None
    try:
        return _model_sentiment(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo Pysentimiento.") from error
        return _lexical_sentiment(transcription), "fallback", type(error).__name__
